# Tunisian Arabic Speech-to-English Translation

Compare **audio → ASR → NLLB text translation** with **direct audio → Whisper translation**. English is selected because TEDxTN supplies English references. This notebook requires locally supplied talk WAV files; metadata-only cells remain runnable without audio. Do not treat generated metrics as complete unless a non-smoke run has covered the intended held-out test set.

In [ ]:
from pathlib import Path
import json, os, random, subprocess, sys, time, gc
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data_utils import read_csv_rows, align_source_translation, inspect_audio_directory, load_audio_segment

SEED = 42
random.seed(SEED)
DATA_DIR = ROOT / 'data' / 'metadata'
AUDIO_DIR = Path(os.environ.get('TEDXTN_AUDIO_DIR', ROOT / 'data' / 'audio')).expanduser().resolve()
if not all((DATA_DIR / f'{split}.csv').is_file() for split in ('train','dev','test')):
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'inspect_dataset.py'), '--download-metadata'], check=True)
RUN_SMOKE_TEST = True
RUN_TRANSLATION = False
ALIGNMENT_REVIEWED = False
ASR_MODEL_ID = str(ROOT / 'results' / 'models' / 'asr-best') if (ROOT / 'results' / 'models' / 'asr-best').exists() else 'openai/whisper-small'
ASR_MODEL_REVISION = '973afd24965f72e36ca33b3055d56a652f456b4d'
DIRECT_MODEL_ID = 'openai/whisper-small'
DIRECT_MODEL_REVISION = '973afd24965f72e36ca33b3055d56a652f456b4d'
MT_MODEL_ID = 'facebook/nllb-200-distilled-600M'
MT_MODEL_REVISION = 'f8d333a098d19b4fd9a8b18f94170487ad3f821d'
print('ASR stage:', ASR_MODEL_ID)
print('Direct speech translation:', DIRECT_MODEL_ID)
print('Text MT:', MT_MODEL_ID, '(aeb_Arab → eng_Latn)')

In [ ]:
source_rows = read_csv_rows(DATA_DIR / 'test.csv')
target_rows = read_csv_rows(DATA_DIR / 'test.en.csv')
test_pairs, alignment_report = align_source_translation(source_rows, target_rows)
test_pairs = [pair for pair in test_pairs if pair['translation'].strip()]
alignment_df = pd.DataFrame(test_pairs)
if not alignment_df.empty:
    alignment_review = pd.concat([
        alignment_df.groupby('audio_filename', group_keys=False).head(2),
        alignment_df.groupby('audio_filename', group_keys=False).tail(1)
    ]).drop_duplicates(['audio_filename','ordinal']).sort_values(['audio_filename','ordinal'])
else:
    alignment_review = alignment_df
print(json.dumps(alignment_report, ensure_ascii=False, indent=2))
print('Non-empty candidate pairs:', len(test_pairs))
display(alignment_review[['audio_filename','ordinal','start_time','end_time',
    'translation_start_time','translation_end_time','source_overlap_ratio',
    'target_overlap_ratio','transcription','translation']])
print('Review these examples across every test talk. Pairing is ordinal within talk, with >=50% overlap required for both intervals; this remains a candidate heuristic, not authoritative sentence alignment.')

In [ ]:
audio_audit = inspect_audio_directory(AUDIO_DIR, {'test': source_rows})
audio_audit_path = ROOT / 'results' / 'metrics' / 'audio_inventory_test.json'
audio_audit_path.parent.mkdir(parents=True, exist_ok=True)
audio_audit_path.write_text(json.dumps(audio_audit, ensure_ascii=False, indent=2), encoding='utf-8')
AUDIO_READY = audio_audit['splits']['test']['ready'] and bool(test_pairs)
print(f"Test talk WAV files: {audio_audit['present_readable_files']}/{audio_audit['expected_talks']}")
print('Missing files:', audio_audit['missing_files'])
print('Invalid files:', audio_audit['invalid_files'])
print('Test intervals beyond audio:', audio_audit['out_of_bounds_segments'])

## Models and comparability

The two-stage path uses the adapted ASR checkpoint if the ASR notebook saved it; otherwise it uses zero-shot Whisper-small. It then translates Arabic text using NLLB-200 distilled 600M with its Tunisian Arabic (`aeb_Arab`) and English (`eng_Latn`) language codes. The direct path uses Whisper's speech `translate` task. Whisper multilingual Arabic support and NLLB language coverage do not guarantee Tunisian dialect quality. The same candidate test audio and English references are used for both approaches. Do not tune on test references.

In [ ]:
if RUN_TRANSLATION:
    if not AUDIO_READY:
        raise RuntimeError('Translation evaluation requires audio for all candidate test talks.')
    if not ALIGNMENT_REVIEWED:
        raise RuntimeError('Manually verify the displayed source/translation candidate pairs, then set ALIGNMENT_REVIEWED=True.')
    import torch
    from transformers import (pipeline, AutoModelForSeq2SeqLM, AutoTokenizer)
    from sacrebleu.metrics import BLEU, CHRF
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    device = 0 if torch.cuda.is_available() else -1
    pair_limit = 8 if RUN_SMOKE_TEST else None
    evaluation_pairs = test_pairs[:pair_limit] if pair_limit else test_pairs
    references = [row['translation'] for row in evaluation_pairs]

    # Approach 1: ASR followed by text MT.
    asr_kwargs = {'model': ASR_MODEL_ID, 'device': device}
    asr_revision_used = None
    if ASR_MODEL_ID == 'openai/whisper-small':
        asr_kwargs['revision'] = ASR_MODEL_REVISION
        asr_revision_used = ASR_MODEL_REVISION
    asr = pipeline('automatic-speech-recognition', **asr_kwargs)
    transcripts, asr_seconds = [], 0.0
    for row in evaluation_pairs:
        wav = load_audio_segment(AUDIO_DIR, row['audio_filename'], row['start_time'], row['end_time'])
        started = time.perf_counter()
        output = asr({'raw': wav, 'sampling_rate': 16000}, generate_kwargs={'language':'ar','task':'transcribe'})
        asr_seconds += time.perf_counter() - started
        transcripts.append(output['text'])
    del asr
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    source_language, target_language = 'aeb_Arab', 'eng_Latn'
    tokenizer = AutoTokenizer.from_pretrained(MT_MODEL_ID, revision=MT_MODEL_REVISION, src_lang=source_language)
    mt_model = AutoModelForSeq2SeqLM.from_pretrained(
        MT_MODEL_ID, revision=MT_MODEL_REVISION,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
    ).to('cuda' if torch.cuda.is_available() else 'cpu').eval()
    forced_bos_id = tokenizer.convert_tokens_to_ids(target_language)
    if forced_bos_id == tokenizer.unk_token_id:
        raise ValueError(f'{MT_MODEL_ID} tokenizer has no target code {target_language!r}')
    mt_hypotheses, mt_seconds = [], 0.0
    for transcript in transcripts:
        encoded = tokenizer(transcript, return_tensors='pt', truncation=True, max_length=512)
        encoded = {key: value.to(mt_model.device) for key, value in encoded.items()}
        started = time.perf_counter()
        with torch.inference_mode():
            generated = mt_model.generate(**encoded, forced_bos_token_id=forced_bos_id, max_new_tokens=256)
        mt_seconds += time.perf_counter() - started
        mt_hypotheses.append(tokenizer.batch_decode(generated, skip_special_tokens=True)[0])
    del mt_model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Approach 2: speech translation directly from audio.
    direct = pipeline('automatic-speech-recognition', model=DIRECT_MODEL_ID,
        revision=DIRECT_MODEL_REVISION, device=device)
    direct_hypotheses, direct_seconds = [], 0.0
    for row in evaluation_pairs:
        wav = load_audio_segment(AUDIO_DIR, row['audio_filename'], row['start_time'], row['end_time'])
        started = time.perf_counter()
        output = direct({'raw': wav, 'sampling_rate': 16000}, generate_kwargs={'language':'ar','task':'translate'})
        direct_seconds += time.perf_counter() - started
        direct_hypotheses.append(output['text'])
    del direct
    gc.collect()

    bleu, chrf = BLEU(), CHRF()
    results = {
        'status': 'smoke_test' if RUN_SMOKE_TEST else 'full_test',
        'n_examples': len(references),
        'source_rows_not_evaluated': len(source_rows) - len(evaluation_pairs),
        'empty_translation_rows_excluded': sum(not row['translation'].strip() for row in target_rows),
        'asr_model': ASR_MODEL_ID, 'asr_model_revision': asr_revision_used,
        'mt_model': MT_MODEL_ID, 'mt_model_revision': MT_MODEL_REVISION,
        'direct_model': DIRECT_MODEL_ID, 'direct_model_revision': DIRECT_MODEL_REVISION,
        'seed': SEED, 'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'torch_version': torch.__version__, 'transformers_version': __import__('transformers').__version__,
        'two_stage': {'bleu': bleu.corpus_score(mt_hypotheses, [references]).score,
                      'chrf': chrf.corpus_score(mt_hypotheses, [references]).score,
                      'asr_seconds': asr_seconds, 'mt_seconds': mt_seconds},
        'direct': {'bleu': bleu.corpus_score(direct_hypotheses, [references]).score,
                   'chrf': chrf.corpus_score(direct_hypotheses, [references]).score,
                   'inference_seconds': direct_seconds},
        'sacrebleu_signature': str(bleu.get_signature()),
        'alignment': alignment_report
    }
    output_dir = ROOT / 'results' / 'metrics'
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / 'translation_results.json').write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
    examples = pd.DataFrame({'audio_filename':[r['audio_filename'] for r in evaluation_pairs],
        'start_time':[r['start_time'] for r in evaluation_pairs], 'end_time':[r['end_time'] for r in evaluation_pairs],
        'reference': references, 'asr_transcript': transcripts,
        'two_stage': mt_hypotheses, 'direct': direct_hypotheses})
    examples_dir = ROOT / 'results' / 'qualitative_examples'
    examples_dir.mkdir(parents=True, exist_ok=True)
    examples.to_csv(examples_dir / 'translation_test_predictions.csv', index=False, encoding='utf-8')
    display(examples.head(10))
    print(json.dumps(results, ensure_ascii=False, indent=2))
else:
    print('Model run skipped by default. Set RUN_TRANSLATION=True only after verifying alignment and audio permissions.')

## Reporting and limitations

BLEU/chrF values from a smoke test are debugging signals only. Save model IDs, the exact evaluated segment IDs, metric signatures, timing, package versions, and the complete predictions. Inspect code-switching, names, disfluencies, and translation alignment errors. The paper's fine-tuned Whisper-large-v3 test BLEU (25.68) is a published reference, not a result from this notebook. Dataset reference: Bougares et al. (2025), [ACL Anthology](https://aclanthology.org/2025.arabicnlp-main.22/).